In [3]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("source.csv")

# Convert 'Datetime' column from UTC to UTC+6
df['Datetime'] = pd.to_datetime(df['Datetime']).dt.tz_localize('UTC').dt.tz_convert('Etc/GMT-6')

# Separate Product A and Product B
product_a = df[df['Name'] == 'ProductA'].copy()
product_b = df[df['Name'] == 'ProductB'].copy()

# Merge Product B with Product A on 'Datetime' to find matching prices
merged = pd.merge(
    product_b,
    product_a[['Datetime', 'Price']],
    on='Datetime',
    how='left',
    suffixes=('_B', '_A')
)

# Function to calculate total
def calculate_total(row, is_product_a=True):
    if is_product_a:
        price = row['Price']
        if row['Purity'] == 'Impure':
            price *= 0.75
        return row['Amount'] * price
    else:
        price_diff = row['Price_B'] - row['Price_A']
        if row['Purity'] == 'Impure':
            price_diff *= 0.75
        return row['Amount'] * price_diff

# Apply total calculation
product_a['Total'] = product_a.apply(lambda row: calculate_total(row, is_product_a=True), axis=1)
merged['Total'] = merged.apply(lambda row: calculate_total(row, is_product_a=False), axis=1)

# Combine and clean up
final_df = pd.concat([
    product_a[['Name', 'Datetime', 'Amount', 'Price', 'Purity', 'Total']],
    merged[['Name', 'Datetime', 'Amount', 'Price_B', 'Purity', 'Total']].rename(columns={'Price_B': 'Price'})
]).sort_values('Datetime')

# Save to result.csv
final_df.to_csv("result.csv", index=False)
